# Multi-Hop Retrieval — Full System Evaluation

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Ensure Google Drive `musique_data/` contains:
   - `musique_ans_v1.0_train.jsonl`
   - `musique_ans_v1.0_dev.jsonl`
   - `model1_complement.pt`  (trained ComplementEncoder)
   - `model2_scorer.pt`      (trained ColBERTScorer — best checkpoint)

**What this does:**
Runs full Recall@K comparison across all 2,417 MuSiQue dev queries:
- **MDR baseline** — BM25 + dense retrieval
- **Graph + Cosine** — graph traversal, no trained models
- **Graph + M1 + M2** — ComplementEncoder + ColBERT MaxSim (your full system)

In [ ]:
# ── [EDIT IF NEEDED] ───────────────────────────────────────────────────────────
REPO_URL       = "https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git"
REPO_NAME      = "multihop-retrieval"
DRIVE_DATA_DIR = "/content/drive/MyDrive/musique_data"
# ───────────────────────────────────────────────────────────────────────────────

In [ ]:
# 1. Verify GPU
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime → Change runtime type → T4 GPU")

print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("CUDA OK")

In [ ]:
# 2. Clone repo + install dependencies
import os

if not os.path.isdir(f"/content/{REPO_NAME}"):
    os.system(f"git clone {REPO_URL} /content/{REPO_NAME}")
else:
    os.system(f"cd /content/{REPO_NAME} && git pull")

os.chdir(f"/content/{REPO_NAME}/retrieval")
print("Working dir:", os.getcwd())

# Colab already has torch — only install missing packages
os.system("pip install -q 'transformers>=4.35.0' 'rank_bm25>=0.2.2' 'faiss-gpu' 'sentence-transformers>=2.2.0'")
print("Dependencies installed")

In [ ]:
# 3. Mount Google Drive + verify all required files
from google.colab import drive
drive.mount('/content/drive')

import os, time
time.sleep(3)  # allow FUSE to sync

required = [
    "musique_ans_v1.0_train.jsonl",
    "musique_ans_v1.0_dev.jsonl",
    "model1_complement.pt",
    "model2_scorer.pt",
]
all_ok = True
for fname in required:
    path = f"{DRIVE_DATA_DIR}/{fname}"
    if os.path.exists(path):
        print(f"  {fname}: {os.path.getsize(path)/1e6:.1f} MB  OK")
    else:
        print(f"  {fname}: MISSING — upload to My Drive/musique_data/")
        all_ok = False

if not all_ok:
    raise FileNotFoundError("Some required files are missing from Drive. See above.")
print("\nAll files found — ready to evaluate")

In [ ]:
# 4. Symlink data + copy model checkpoints to expected paths
import os, shutil

os.makedirs("data/musique", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("cache", exist_ok=True)

# MuSiQue data — symlink (avoid copying 260MB)
for fname in ["musique_ans_v1.0_train.jsonl", "musique_ans_v1.0_dev.jsonl"]:
    src = f"{DRIVE_DATA_DIR}/{fname}"
    dst = f"data/musique/{fname}"
    if not os.path.exists(dst):
        os.symlink(src, dst)
    print(f"  {fname}: OK")

# Model checkpoints — copy to models/
for ckpt in ["model1_complement.pt", "model2_scorer.pt"]:
    src = f"{DRIVE_DATA_DIR}/{ckpt}"
    dst = f"models/{ckpt}"
    if not os.path.exists(dst):
        print(f"  Copying {ckpt} ({os.path.getsize(src)/1e6:.0f} MB) ...", flush=True)
        shutil.copy(src, dst)
    print(f"  {ckpt}: OK")

print("\nAll paths ready")

---
## Run Full Evaluation

Evaluates all 2,417 MuSiQue dev queries (~48K chunks).  
Expected time: **~20 min** on T4 (dense embedding dominates).

In [ ]:
# 5. Run full system comparison
import os
os.chdir(f"/content/{REPO_NAME}/retrieval")

result_file = "/content/eval_results.txt"
ret = os.system(f"python run_full_system.py 2>&1 | tee {result_file}")
print(f"\nExit code: {ret}")

In [ ]:
# 6. Display results summary
import re

with open("/content/eval_results.txt") as f:
    log = f.read()

# Print just the comparison table (lines containing Recall or system names)
lines = log.split("\n")
in_table = False
for line in lines:
    if any(kw in line for kw in ["====", "System", "Recall", "MDR", "Graph", "FULL", "----", "R@"]):
        in_table = True
    if in_table:
        print(line)

In [ ]:
# 7. Save results to Google Drive
import shutil
shutil.copy("/content/eval_results.txt", f"{DRIVE_DATA_DIR}/eval_results.txt")
print(f"Results saved to Drive: {DRIVE_DATA_DIR}/eval_results.txt")